# 🍓🐧 Benchmark 01: PyTorch Eager (`01_pytorch`) — UnB Metodologia Científica (Trabalho 2)

Bem-vindos ao nosso laboratório experimental no Google Colab Pro! ✨ Neste notebook, executamos o **Baseline de Alta Abstração em PyTorch Eager** (`implementations/01_pytorch`) para um modelo **Decoder-Only Transformer ( estilo GPT, 10.788.929 parâmetros)** treinado no corpus *TinyShakespeare* (`vocab_size = 65`).

### 🎯 Invariantes Metodológicos (Controle Rigoroso de Variáveis)
- **Semente Global Determinística (`SEED`):** `998244353` (mesmos pesos iniciais $\theta_0$ e mesmos mini-batches $X_t, Y_t$ em todas as 3 implementações!)
- **Precisão Numérica:** `FP32` estrito (`tf32` desabilitado) para paridade matemática bit-a-bit entre PyTorch, Triton e CUDA C++.
- **Arquitetura Invariante:** `n_layer = 6`, `n_head = 6`, `n_embd = 384`, `block_size = 256`, `batch_size = 64`, `dropout = 0.0`.
- **Protocolo Experimental:** Sanity Check ($|L_0 - L_0^{\text{ref}}| < 10^{-4}$ e $|L_1 - L_1^{\text{ref}}| < 10^{-4}$) $\rightarrow$ 50 steps de Warm-up $\rightarrow$ Reload $\theta_0$ $\rightarrow$ 5000 steps de Treino Cronometrado + Inferência Autoregressiva ($B=1$ e $B=64$, 1000 novos tokens).

## 🛠️ Célula 1: Verificação da GPU & Setup do Repositório GitHub 🍓

Vamos inspecionar a GPU alocada pelo Google Colab Pro (`!nvidia-smi`) e garantir que o repositório experimental esteja clonado e pronto no diretório de trabalho!

In [ ]:
!nvidia-smi

import os
from pathlib import Path

REPO_URL = "https://github.com/<SEU_USUARIO>/<SEU_REPO>.git"
REPO_DIR = Path("trabalho_2")

if Path("common/generate_golden.py").exists() and Path("implementations/01_pytorch/benchmark.py").exists():
    print("🍓 [Chuu Lab] Repositório já detectado no diretório atual:", Path.cwd())
elif (REPO_DIR / "common/generate_golden.py").exists():
    os.chdir(REPO_DIR)
    print("🍓 [Chuu Lab] Entrando no diretório existente do repositório:", Path.cwd())
else:
    print(f"🐧 [Chuu Lab] Clonando repositório de {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}
    if REPO_DIR.exists():
        os.chdir(REPO_DIR)
    print("🍓 [Chuu Lab] Diretório ativo:", Path.cwd())


## 🌟 Célula 2: Geração da Referência Dourada (`SEED = 998244353`) & Sanity Check 🐧

Aqui geramos os artefatos dourados determinísticos em `/tmp/golden_gpt` contendo os pesos iniciais $\theta_0$, os 5000 batches de treino pré-amostrados e as perdas de referência exatas `loss_step_0` e `loss_step_1`.

In [ ]:
!python3 -m common.generate_golden --output-dir /tmp/golden_gpt --num-steps 5000 --eval-iters 50 --seed 998244353

import json
from pathlib import Path

golden_meta = json.loads(Path("/tmp/golden_gpt/golden_meta.json").read_text(encoding="utf-8"))
print("\n🍓 === Resumo do Golden Meta (SEED = 998244353) === 🐧")
print(json.dumps({
    "seed": golden_meta["seed"],
    "num_parameters": golden_meta["num_parameters"],
    "vocab_size": golden_meta["vocab_size"],
    "loss_step_0": golden_meta["loss_step_0"],
    "loss_step_1": golden_meta["loss_step_1"],
}, indent=2))


## 🚀 Célula 3: Execução do Benchmark Completo (`01_pytorch` — 5000 Steps + Inferência $B=1$ e $B=64$) 🍓

Executamos o protocolo experimental completo para a implementação PyTorch Eager e salvamos as métricas em `results_01_pytorch.json`.

In [ ]:
!python3 implementations/01_pytorch/benchmark.py --golden-dir /tmp/golden_gpt --output-json results_01_pytorch.json --max-iters 5000 --warmup-iters 50 --eval-interval 500 --eval-iters 50 --gen-tokens 1000


## 📊 Célula 4: Inspeção das Métricas e Download Automático do JSON 🐧

Exibimos o resumo científico das variáveis dependentes (Throughput, Tempo Wall-Clock, Sanity Check, Pico de VRAM e SLOC) e iniciamos o download automático de `results_01_pytorch.json`.

In [ ]:
import json
from pathlib import Path

result_path = Path("results_01_pytorch.json")
report = json.loads(result_path.read_text(encoding="utf-8"))

summary_view = {
    "implementation": report.get("implementation"),
    "gpu_name": report.get("gpu_name"),
    "sanity_check": report.get("sanity_check"),
    "train_time_s": report.get("train_time_s"),
    "train_tokens_per_sec": report.get("train_tokens_per_sec"),
    "infer_b1_tokens_per_sec": report.get("infer_b1_tokens_per_sec"),
    "infer_b1_latency_ms_per_token": report.get("infer_b1_latency_ms_per_token"),
    "infer_b64_tokens_per_sec": report.get("infer_b64_tokens_per_sec"),
    "peak_vram_mb": report.get("peak_vram_mb"),
    "final_train_loss": report.get("final_train_loss"),
    "final_val_loss": report.get("final_val_loss"),
    "sloc_core": report.get("sloc_core"),
    "sloc_harness": report.get("sloc_harness"),
    "sloc_total": report.get("sloc_total"),
}

print("🍓 === Resumo Final: 01_pytorch === 🐧")
print(json.dumps(summary_view, indent=2))

try:
    from google.colab import files
    files.download("results_01_pytorch.json")
except ImportError:
    print("🐧 Execução fora do Google Colab detectada; download automático ignorado.")
